💡 **Environment:** `clamp-analyses`

# Supplementary Figure 7: ARCHS4 drug-disease canonical model, complete method comparison

ROC/AUROC/AUPRC comparison of ARCHS4, recount2, GTEx, and gene-based drug-disease prediction, plus every paired per-tissue method comparison not already reported in main Figure 3.

In [1]:
suppressPackageStartupMessages({
  library(here)
})

In [2]:
source(here("scripts/panels/style.R"))
NM_HEIGHT <- 110
set.seed(123)
out <- here("output/99_panels/supp7/source_data")
dir.create(out, recursive=TRUE, showWarnings=FALSE)
# The upstream scores are a pandas pickle. Convert and calculate ROC values here
# so this notebook produces all of its own source data before drawing the figure.
python <- file.path(dirname(dirname(R.home())), "bin", "python")
if (!file.exists(python)) python <- Sys.which("python3")
stopifnot(nzchar(python), file.exists(python))
export_code <- c(
  "import sys, pathlib, pandas as pd",
  "from sklearn.metrics import roc_curve, roc_auc_score",
  "source, out = pathlib.Path(sys.argv[1]), pathlib.Path(sys.argv[2])",
  "scores = pd.read_pickle(source)",
  "names = {'module_based_archs4':'ARCHS4', 'module_based_recount2':'recount2', 'module_based_gtex':'GTEx', 'gene_based':'Gene-based'}",
  "scores['model'] = scores.method.map(names)",
  "assert set(scores.model) == set(names.values())",
  "scores.to_csv(out / 'prediction_scores.csv', index=False)",
  "curves, summary = [], []",
  "for model in names.values():",
  "    sub = scores[scores.model == model]",
  "    fpr, tpr, thresholds = roc_curve(sub.true_class, sub.score)",
  "    curves.append(pd.DataFrame({'model':model, 'fpr':fpr, 'tpr':tpr, 'threshold':thresholds}))",
  "    summary.append({'model':model, 'AUROC':roc_auc_score(sub.true_class, sub.score), 'n_pairs':len(sub), 'n_positive':int(sub.true_class.sum())})",
  "pd.concat(curves).to_csv(out / 'roc_coordinates.csv', index=False)",
  "pd.DataFrame(summary).to_csv(out / 'performance_summary.csv', index=False)"
)
export_file <- tempfile(fileext=".py")
writeLines(export_code, export_file)
score_pickle <- here("output/03_model_biology/02_archs4/02_drug_diseases_canonical/aggregate/max/predictions_results_aggregated.pkl")
stopifnot(file.exists(score_pickle))
export_status <- system2(python, shQuote(c(export_file, score_pickle, out)))
unlink(export_file)
stopifnot(export_status == 0L)
scores <- fread(file.path(out,"prediction_scores.csv"));roc <- fread(file.path(out,"roc_coordinates.csv"));perf <- fread(file.path(out,"performance_summary.csv"))
base <- here("output/03_model_biology/02_archs4/02_drug_diseases_canonical/three_model_comparison/data")
paths <- setNames(file.path(base,paste0(c("per_tissue_metrics","max_aggregate_reference","paired_tissue_tests","ordering_stability"),".csv")),
                  c("per_tissue_metrics","max_aggregate_reference","paired_tissue_tests","ordering_stability"))
nm_sources(7,paths)
tissue <- fread(paths[1]);tests <- fread(paths[3]);agg <- fread(paths[2])
order <- c("ARCHS4","recount2","GTEx","Gene-based")
colours <- unlist(yaml::read_yaml(here("config.yaml"))$COMPENDIUM_COLORS)[c("ARCHS4","recount2","GTEx","Gene-based")]
scores[,model:=factor(model,levels=order)];roc[,model:=factor(model,levels=order)]
# Compact two-by-two layout: ROC and score distribution on top, summary dots below.
tight <- theme(plot.margin=margin(1,2,1,1,"mm"))
A <- ggplot(scores,aes(score/1e5,colour=model))+
  geom_density(linewidth=.45,adjust=1.2,key_glyph="path")+
  scale_colour_manual(values=colours,name=NULL)+
  scale_x_continuous(breaks=1:4)+scale_y_continuous(expand=expansion(mult=c(0,.05)))+
  labs(x=expression("Aggregated drug–disease score ("*10^displaystyle(5)*")"),y="Density")+nm_theme()+tight+
  theme(legend.position=c(.02,.98),legend.justification=c(0,1),legend.background=element_blank(),
        legend.key.height=unit(2.6,"mm"),legend.key.width=unit(4,"mm"),legend.spacing.y=unit(0,"mm"))
roc_labels <- setNames(sprintf("%s (AUROC %.3f)",perf$model,perf$AUROC),perf$model)
B <- ggplot(roc,aes(fpr,tpr,colour=model))+geom_path(linewidth=.45)+
  geom_abline(slope=1,intercept=0,linetype="dashed",colour="grey60",linewidth=.3)+
  scale_colour_manual(values=colours,labels=roc_labels,name=NULL)+
  scale_x_continuous(limits=c(0,1),breaks=seq(0,1,.5),expand=expansion(mult=.02))+
  scale_y_continuous(limits=c(0,1),breaks=seq(0,1,.5),expand=expansion(mult=.02))+
  coord_equal()+labs(x="False positive rate",y="True positive rate")+nm_theme()+tight+
  theme(legend.position=c(1.02,.02),legend.justification=c(1,0),legend.background=element_blank(),
        legend.key.height=unit(2.6,"mm"),legend.key.width=unit(4,"mm"),legend.spacing.y=unit(0,"mm"))
perf[,model:=factor(model,levels=rev(order))]
C <- ggplot(perf,aes(AUROC,model,colour=model))+geom_vline(xintercept=.5,linetype="dashed",colour="grey60",linewidth=.3)+
  geom_point(size=1.8)+geom_text(aes(label=sprintf("%.3f",AUROC)),hjust=-.35,size=5.5/.pt,colour="black")+
  scale_colour_manual(values=colours,guide="none")+scale_x_continuous(limits=c(.48,.69),breaks=seq(.5,.65,.05))+
  labs(x="Max-aggregated AUROC",y=NULL)+nm_theme()+tight
tests[,comparison:=paste(group1,"vs",group2)]
tests[,label:=paste0("q = ",nm_q(q_value_bh))]
D <- ggplot(tests,aes(-log10(q_value_bh),factor(comparison,levels=rev(comparison))))+
  geom_vline(xintercept=-log10(.05),linetype="dashed",colour="grey60",linewidth=.3)+
  geom_point(size=1.5,colour="#1B4D43")+geom_text(aes(x=6.5,label=label),hjust=0,size=5.5/.pt)+
  scale_x_continuous(breaks=seq(0,8,2),limits=c(0,8.5),expand=expansion(mult=c(.03,.02)))+
  labs(x=expression("−log"[displaystyle(10)]*"(BH-adjusted "*italic(q)*")"),y=NULL)+nm_theme()+tight
# AUROC comparison is in main Figure 3H; preserve every remaining metric in source data.
pages <- list(plot_grid(plot_grid(nm_tag(A,"A"),nm_tag(B,"B"),ncol=2),
                        plot_grid(nm_tag(C,"C"),nm_tag(D,"D"),ncol=2,rel_widths=c(.8,1.2)),
                        ncol=1,rel_heights=c(1.25,.9)))
titles <- "Drug–disease prediction and complete method comparisons"
if("auprc" %in% names(tissue)) {
  tissue[,method:=factor(method,levels=order)]
  E <- ggplot(tissue,aes(method,auprc,fill=method))+geom_boxplot(width=.55,outlier.shape=NA,alpha=.5,linewidth=.3)+
    geom_point(position=position_jitter(width=.08),size=.7,alpha=.6)+
    geom_point(data=agg,aes(method,auprc),shape=23,fill="white",size=2)+
    scale_fill_manual(values=colours,guide="none")+labs(x=NULL,y="AUPRC per GTEx tissue")+nm_theme()
  F <- ggplot(tissue,aes(auroc,auprc,colour=method))+geom_point(size=1,alpha=.75)+
    scale_colour_manual(values=colours,name="Method")+labs(x="AUROC per tissue",y="AUPRC per tissue")+nm_theme()+theme(legend.position="bottom")
  # Full 49-tissue values remain readable as two horizontal panels.
  tissue[,tissue:=factor(tissue,levels=rev(sort(unique(tissue))))]
  L <- ggplot(tissue,aes(auroc,tissue,colour=method))+geom_point(position=position_dodge(width=.55),size=.9)+
    scale_colour_manual(values=colours,name="Method")+labs(x="AUROC",y=NULL)+nm_theme()+theme(axis.text.y=element_text(size=5),legend.position="bottom")
  M <- ggplot(tissue,aes(auprc,tissue,colour=method))+geom_point(position=position_dodge(width=.55),size=.9)+
    scale_colour_manual(values=colours,name="Method")+labs(x="AUPRC",y=NULL)+nm_theme()+theme(axis.text.y=element_text(size=5),legend.position="bottom")
  pages <- c(pages,list(plot_grid(nm_tag(E,"E"),nm_tag(F,"F"),ncol=2),
                       plot_grid(nm_tag(L,"G"),nm_tag(M,"H"),ncol=2)))
  titles <- c(titles,"Complementary precision–recall metrics","Tissue-specific performance")
}
nm_export(pages,7,titles,height=110)

[1] "/home/msubirana/Documents/pivlab/clamp-analyses-final_figures/output/99_panels/supp7/source_data"

Supplementary Figure 7: 1 pages, 180 x 110 mm
